# Part 2: MFCC Feature Extraction Benchmark

This notebook tests the three standalone profiling scripts using the same audio files and presents their outputs in a comparable table.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

AUDIO_FILES = [Path("../data/test1.wav"), Path("../data/test2.wav")]
N_RUNS = 7

for path in AUDIO_FILES:
    if not path.exists():
        raise FileNotFoundError(f"Missing audio file: {path.resolve()}")

## 2. Load the three implementations

In [ ]:
import profile_librosa
import profile_scipy
import profile_pytorch

BACKENDS = {
    "Librosa": profile_librosa,
    "SciPy/NumPy": profile_scipy,
    "PyTorch CPU": profile_pytorch,
}

## 3. Verify output shapes and numerical agreement

In [ ]:
comparison_rows = []
outputs = {}

for audio_path in AUDIO_FILES:
    for name, module in BACKENDS.items():
        signal = module.load_audio(audio_path)
        output = module.extract_mfcc(signal)
        outputs[(audio_path.name, name)] = output
        comparison_rows.append({
            "file": audio_path.name,
            "backend": name,
            "shape": tuple(output.shape),
            "dtype": str(output.dtype),
        })

display(pd.DataFrame(comparison_rows))

accuracy_rows = []
for audio_path in AUDIO_FILES:
    reference = outputs[(audio_path.name, "Librosa")].numpy()
    for name in BACKENDS:
        candidate = outputs[(audio_path.name, name)].numpy()
        frames = min(reference.shape[-1], candidate.shape[-1])
        diff = candidate[..., :frames] - reference[..., :frames]
        accuracy_rows.append({
            "file": audio_path.name,
            "backend": name,
            "MAE": float(np.mean(np.abs(diff))),
            "RMSE": float(np.sqrt(np.mean(diff ** 2))),
            "max_abs_error": float(np.max(np.abs(diff))),
        })

display(pd.DataFrame(accuracy_rows).round(6))

## 4. Runtime and RTF benchmark

In [ ]:
benchmark_rows = []

for audio_path in AUDIO_FILES:
    for name, module in BACKENDS.items():
        signal = module.load_audio(audio_path)
        duration = module.get_audio_duration(audio_path)
        _, wall_times, cpu_times = module.profile_timing(signal, N_RUNS)
        avg_wall = float(np.mean(wall_times))
        avg_cpu = float(np.mean(cpu_times))
        benchmark_rows.append({
            "file": audio_path.name,
            "backend": name,
            "runs": N_RUNS,
            "avg_wall_ms": avg_wall * 1000,
            "std_wall_ms": float(np.std(wall_times)) * 1000,
            "avg_cpu_ms": avg_cpu * 1000,
            "cpu_percent_one_core": 100 * avg_cpu / avg_wall if avg_wall else 0,
            "RTF": avg_wall / duration,
        })

benchmark_table = pd.DataFrame(benchmark_rows)
display(benchmark_table.round(5))

## 5. Standalone commands

Run these commands from the `part2-feature-extraction` directory:

```bash
python profile_librosa.py ../data/test1.wav 50
python profile_scipy.py ../data/test1.wav 50
python profile_pytorch.py ../data/test1.wav 50
```

For memory profiling, use Memray as documented at the top of each script.